In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

from datetime import datetime, timedelta

In [ ]:
# %run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Sentiment/Functions.ipynb"
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl', compression=None)
df_companies=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SPXCompaniesByMarketCap.pkl")

In [ ]:
df_articles.index = df_articles.Date
df_articles = df_articles["2020-06-01":"2022-06-01"]

In [ ]:
df_articles

In [ ]:
df_articles["Hours"]=df_articles["Time"].dt.strftime('%Y-%m-%d %H')


In [ ]:
df_articles.index = df_articles.Time

In [ ]:
df_articles.index = df_articles.index.to_period(freq='T')
df_articles.index.unique()

In [ ]:
df_articles["Articles"].apply(lambda n: len(n.split())).sum()


In [ ]:
df_articles["Date"].drop_duplicates()

In [ ]:
df_companies

In [ ]:
# New words and values
new_words = {
    'crushes': 100,
    'beats': 5,
    'misses': -5,
    'trouble': -10,
    'falls': -100,
    "dividends":100,
    "dividend":100,
    "earning":100,
    "ipo":100,
    "alliance":100,
    "enhance":100,
    "enhanc":100,
    "down":-100,
    "acquisition":100, 
    "up":100,
    "great":100,
    "expand":10,
    "collaboration":100,
    "fda clearance":100
}
# Instantiate the sentiment intensity analyzer with the existing lexicon
vader = SentimentIntensityAnalyzer()
# Update the lexicon
vader.lexicon.update(new_words)

In [ ]:
#  df_companies.loc["GM", "2020-06-16"].marketCap


In [ ]:
rows_list = []
for date, row in df_articles.T.iteritems():
    Hour = row["Hour"]
    Time = row["Time"]
    Date = row["Date"]
    print(date)
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    ############# MarketCap #############
    while(((row["Company"], Date) in df_companies.index)==False):
        x = Date
        Date = (datetime.strptime(x, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')
    MarketCap = df_companies.loc[row["Company"], Date].marketCap
    ############# Sentiment ##############
    ss = vader.polarity_scores(sentence)

    Compound = ss['compound']
    Neg = ss['neg']
    Neu = ss['neu']
    Pos = ss['pos']
    Sentiment=0
    if ((Pos + Neg) != 0):
        Sentiment = MarketCap*((Pos + Neg) / (Pos + Neg + Neu))
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    polarityB = ""
    if (blob.sentiment.polarity > 0 ):
        polarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        polarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        polarityB = "Neutral"
    #################### New Line #######################
    new_row = {"Time":row["Time"], "Date":row["Date"], "Hour": row["Hour"], "sentiment":Sentiment, "compound":Compound, "neg":Neg, "neu":Neu, "pos":Pos, "polarity": Polarity, "subjectivity":Subjectivity, "polarityB":polarityB,"Articles":row["Articles"], "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "MarketCap":MarketCap}
    rows_list.append(new_row)


In [ ]:
date

In [ ]:
df = pd.DataFrame(data=rows_list, columns=["Time", "Date", "Hour", "sentiment", "compound", "neg", "neu", "pos","polarity", "subjectivity" ,"polarityB","Articles","Title","Text", "Company","MarketCap"])

In [ ]:
times = pd.to_datetime(df.Time)

In [ ]:
df

In [ ]:
df_hourly = df.groupby([df.Date, df.Hour]).sum()

In [ ]:
df_hourly[["Date","Hour"]]

In [ ]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTime.pkl")

# Date

In [ ]:
df.loc[df["sentiment"]==0]

In [ ]:
df_articles.loc[df_articles["Company"]=="XOM"].loc["2020-06-01"].Text

In [ ]:
df.groupby("Date").count()["Time"]

In [ ]:
df_sentiment = df.groupby("Date").sum()

In [ ]:
df_sentiment

In [ ]:
df_sentiment["PressReleaseCount"]=df.groupby("Date").count()["Time"]

In [ ]:
df_sentiment["sentiment"] = df_sentiment["sentiment"]/df_sentiment["MarketCap"]
df_sentiment["sentiment"] = df_sentiment["sentiment"]*100
df_sentiment[["sentiment","PressReleaseCount"]].plot(figsize=(19,19))

In [ ]:
vader.polarity_scores("GM'S SOUTH KOREA UNIT: MAY SALES DOWN 40% Y/Y TO 24,778 VEHICLES.")

In [ ]:
df_sentiment.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentDaily.pkl")

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentDaily.pkl")